# LEBER A4 Champion - Evaluasi Akhir pada Test Set (525 Pasien Terkunci)

**Tahap 11: Evaluasi Akhir Satu Kali (One-Time Final Evaluation)**

Notebook ini mengevaluasi model champion arsitektur final **LEBER A4 (Label-Wise Dynamic Router)** pada data uji (**Test Set: 525 pasien**) yang selama seluruh rangkaian kontrol (A0 s.d. A3) dan pencarian hiperparameter dijaga **100% bebas dari kebocoran data (zero test leakage)**.

### Protokol Ilmiah Evaluasi:
1. **Model yang Dievaluasi:**
   - LEBER A4 Seed 42
   - LEBER A4 Seed 52
   - LEBER A4 Seed 62
   - Rata-rata 3 Seed (Mean ± Std)
   - **3-Seed Ensemble** (Fusi rata-rata probabilitas Model Seed 42, 52, dan 62)
2. **Ambang Keputusan (Decision Thresholds):**
   - Menguji **Ambang Default 0.50** (Standar internasional).
   - Menguji **Ambang Terkunci Validation (Val-Locked Tuned)** yang ditentukan secara ketat HANYA dari Validation Set. DILARANG mencari ambang baru pada Test Set.
3. **Metrik Standar Komprehensif:**
   - Macro-F1 (Tuned), Macro-F1 (Default 0.50), Micro-F1, Macro-AUROC, Hamming Loss, Subset Accuracy, serta metrik per-label untuk ke-8 kelas (N, D, G, C, A, H, M, O).
4. **Audit Konsistensi Swap (Pertukaran Mata):**
   - Menguji apakah sifat invarian Δp = 0.0000 dan ekuivarian Δw = 0.0000 tetap terbukti mutlak pada data uji.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device Evaluasi:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 1. Lokasi Dataset ODIR-5K
INPUT_ROOT = Path("/kaggle/input")

test_matches = list(INPUT_ROOT.rglob("test.csv"))
if not test_matches:
    raise FileNotFoundError("Berkas test.csv tidak ditemukan di /kaggle/input!")

TEST_CSV = test_matches[0]
DATASET_DIR = TEST_CSV.parent

# Lokasi validation.csv jika tersedia
val_matches = list(DATASET_DIR.rglob("validation.csv"))
VALID_CSV = val_matches[0] if val_matches else None

# Lokasi citra test
candidate_dirs = [
    DATASET_DIR / "Training Images" / "Training Images",
    DATASET_DIR / "Training Images",
    DATASET_DIR / "Testing Images" / "Testing Images",
    DATASET_DIR / "Testing Images"
]
IMAGE_DIR = None
for cd in candidate_dirs:
    if cd.exists() and (cd / "0_left.jpg").exists():
        IMAGE_DIR = cd
        break

if IMAGE_DIR is None:
    alt_matches = list(DATASET_DIR.rglob("0_left.jpg"))
    if alt_matches:
        IMAGE_DIR = alt_matches[0].parent
    else:
        alt_all = list(INPUT_ROOT.rglob("0_left.jpg"))
        if alt_all:
            IMAGE_DIR = alt_all[0].parent
        else:
            raise FileNotFoundError("Direktori citra tidak ditemukan di dataset!")

print("Dataset Dir:", DATASET_DIR)
print("Image Dir  :", IMAGE_DIR)
print("Validation :", VALID_CSV)

test_df = pd.read_csv(TEST_CSV)
LABELS = ["N", "D", "G", "C", "A", "H", "M", "O"]
LABEL_NAMES = {
    "N": "Normal",
    "D": "Diabetes",
    "G": "Glaucoma",
    "C": "Cataract",
    "A": "AMD",
    "H": "Hypertension",
    "M": "Myopia",
    "O": "Others"
}

print("Test set shape:", test_df.shape)
assert len(test_df) == 525, f"Jumlah pasien test harus tepat 525, ditemukan {len(test_df)}"
assert all(c in test_df.columns for c in LABELS), "Seluruh 8 label harus ada di test.csv"

OUTPUT_DIR = Path("/kaggle/working/leber_a4_final_test_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output Dir :", OUTPUT_DIR)


Device Evaluasi: cuda
GPU: Tesla T4
Dataset Dir: /kaggle/input/datasets/kevinardhana/odir-5k-patient-level-multi-label-fundus-dataset
Image Dir  : /kaggle/input/datasets/kevinardhana/odir-5k-patient-level-multi-label-fundus-dataset/Training Images/Training Images
Validation : /kaggle/input/datasets/kevinardhana/odir-5k-patient-level-multi-label-fundus-dataset/validation.csv
Test set shape: (525, 16)
Output Dir : /kaggle/working/leber_a4_final_test_evaluation


In [2]:
# 2. Deteksi Otomatis Checkpoint 3 Seed (Seed 42, Seed 52, Seed 62)
SEEDS = [42, 52, 62]
CHECKPOINTS = {}

for seed in SEEDS:
    matches = list(INPUT_ROOT.rglob(f"*best*a4*seed{seed}*.pt"))
    if not matches:
        matches = list(INPUT_ROOT.rglob(f"*a4*seed{seed}*.pt"))
    if not matches:
        matches = list(INPUT_ROOT.rglob(f"*seed{seed}*.pt"))
    if not matches:
        matches = list(Path("/kaggle/working").rglob(f"*best*a4*seed{seed}*.pt"))
    if not matches:
        matches = list(Path("/kaggle/working").rglob(f"*seed{seed}*.pt"))
    
    if matches:
        CHECKPOINTS[seed] = matches[0]
        print(f"[OK] Checkpoint Seed {seed} ditemukan: {matches[0]}")
    else:
        print(f"[PERINGATAN] Checkpoint Seed {seed} TIDAK ditemukan di /kaggle/input!")

if len(CHECKPOINTS) == 0:
    print("\nPetunjuk: Unggah ketiga berkas checkpoint .pt (Seed 42, 52, 62) sebagai Kaggle Dataset (misal: 'leber-a4-checkpoints') lalu hubungkan ke notebook ini.")


[OK] Checkpoint Seed 42 ditemukan: /kaggle/input/notebooks/kevinardhana/leber-a5-resnet50-bce-seed-42/leber_a5_auxiliary_supervision_bce_512_seed42/best_leber_a5_auxiliary_supervision_resnet50_bce_512_seed42.pt
[OK] Checkpoint Seed 52 ditemukan: /kaggle/input/notebooks/kevinardhana/leber-a4-labelwise-router-bce-512-seed52/leber_a4_labelwise_router_bce_512_seed52/best_leber_a4_labelwise_router_resnet50_bce_512_seed52.pt
[OK] Checkpoint Seed 62 ditemukan: /kaggle/input/notebooks/kevinardhana/leber-a4-labelwise-router-bce-512-seed62/leber_a4_labelwise_router_bce_512_seed62/best_leber_a4_labelwise_router_resnet50_bce_512_seed62.pt


In [3]:
# 3. Dataset dan Loader Pasangan Citra Fundus Bilateral
eval_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

class FundusPairDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.labels = LABELS

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        left_path = self.image_dir / row["left_image"]
        right_path = self.image_dir / row["right_image"]

        left_image = Image.open(left_path).convert("RGB")
        right_image = Image.open(right_path).convert("RGB")

        if self.transform is not None:
            left_image = self.transform(left_image)
            right_image = self.transform(right_image)

        labels = row[self.labels].values.astype(np.float32)
        return {
            "left_image": left_image,
            "right_image": right_image,
            "labels": torch.tensor(labels, dtype=torch.float32),
            "patient_id": row.get("patient_id", index)
        }

test_dataset = FundusPairDataset(test_df, IMAGE_DIR, eval_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
print("Total batch test loader:", len(test_loader))

valid_loader = None
if VALID_CSV and VALID_CSV.exists():
    valid_df = pd.read_csv(VALID_CSV)
    valid_dataset = FundusPairDataset(valid_df, IMAGE_DIR, eval_transform)
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=16,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    print("Total batch validation loader:", len(valid_loader))


Total batch test loader: 33
Total batch validation loader: 33


In [4]:
# 4. Arsitektur Final LEBER A4 (Label-Wise Dynamic Router)
class LabelWiseRouterBilateralResNet50(nn.Module):
    def __init__(self, num_labels=8, dropout=0.30):
        super().__init__()
        self.num_labels = num_labels
        self.backbone = resnet50(weights=None)
        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        # Tiga Expert Klasifikasi
        self.expert_mono = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim, num_labels)
        )
        self.expert_bilateral = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim * 3, num_labels)
        )

        # Router Dinamis Per-Label (Label-Wise Router MLP)
        self.router_mono = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_labels)
        )
        self.router_bilateral = nn.Sequential(
            nn.Linear(feature_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, num_labels)
        )

    @staticmethod
    def symmetric_features(left_features, right_features):
        sum_features = left_features + right_features
        difference_features = torch.abs(left_features - right_features)
        product_features = left_features * right_features
        return torch.cat([sum_features, difference_features, product_features], dim=1)

    def forward(self, left_image, right_image, return_weights=False):
        left_features = self.backbone(left_image)
        right_features = self.backbone(right_image)

        z_left = self.expert_mono(left_features)
        z_right = self.expert_mono(right_features)

        bilateral_features = self.symmetric_features(left_features, right_features)
        z_bilateral = self.expert_bilateral(bilateral_features)

        s_left = self.router_mono(left_features)
        s_right = self.router_mono(right_features)
        s_bilateral = self.router_bilateral(bilateral_features)

        # Softmax sepanjang dimensi cabang (dim=1) independen untuk tiap label
        scores = torch.stack([s_left, s_right, s_bilateral], dim=1)
        weights = torch.softmax(scores, dim=1)  # [B, 3, num_labels]

        w_left = weights[:, 0, :]       # [B, num_labels]
        w_right = weights[:, 1, :]      # [B, num_labels]
        w_bilateral = weights[:, 2, :]  # [B, num_labels]

        fused_logits = (w_left * z_left + w_right * z_right) + (w_bilateral * z_bilateral)
        if return_weights:
            return fused_logits, weights
        return fused_logits

# Sanity Check Sifat Matematis Equivariance & Invariance
dummy_model = LabelWiseRouterBilateralResNet50(num_labels=8).to(DEVICE)
dummy_model.eval()
with torch.no_grad():
    dummy_l = torch.randn(2, 3, 512, 512, device=DEVICE)
    dummy_r = torch.randn(2, 3, 512, 512, device=DEVICE)
    log_orig, w_orig = dummy_model(dummy_l, dummy_r, return_weights=True)
    log_swap, w_swap = dummy_model(dummy_r, dummy_l, return_weights=True)

dp = torch.max(torch.abs(log_orig - log_swap)).item()
dw_mono = torch.max(torch.abs(w_orig[:, 0, :] - w_swap[:, 1, :])).item()
dw_bilat = torch.max(torch.abs(w_orig[:, 2, :] - w_swap[:, 2, :])).item()
dw = max(dw_mono, dw_bilat)
assert dp < 1e-6, f"Gagal invarian: {dp}"
assert dw < 1e-6, f"Gagal ekuivarian: {dw}"
print(f"Sanity Check Arsitektur: 100% Lolos! (Delta p = {dp:.6f}, Delta w = {dw:.6f})")


Sanity Check Arsitektur: 100% Lolos! (Delta p = 0.000000, Delta w = 0.000000)


In [5]:
# 5. Ambang Keputusan Terkunci dari Validation Set (Zero Test Leakage)
# Ambang historis yang telah terverifikasi dari pelatihan masing-masing seed
RECORDED_VAL_THRESHOLDS = {
    42: np.array([0.06, 0.42, 0.84, 0.85, 0.68, 0.07, 0.87, 0.30], dtype=np.float32),
    52: np.array([0.05, 0.48, 0.92, 0.66, 0.23, 0.55, 0.57, 0.37], dtype=np.float32),
    62: np.array([0.08, 0.28, 0.76, 0.32, 0.91, 0.20, 0.48, 0.57], dtype=np.float32)
}

@torch.no_grad()
def run_inference(model, loader, device):
    model.eval()
    all_targets = []
    orig_probs = []
    swap_probs = []
    orig_weights = []
    swap_weights = []

    for batch in loader:
        left = batch["left_image"].to(device)
        right = batch["right_image"].to(device)
        labels = batch["labels"].cpu().numpy()

        out_orig, w_orig = model(left, right, return_weights=True)
        out_swap, w_swap = model(right, left, return_weights=True)

        all_targets.append(labels)
        orig_probs.append(torch.sigmoid(out_orig).cpu().numpy())
        swap_probs.append(torch.sigmoid(out_swap).cpu().numpy())
        orig_weights.append(w_orig.cpu().numpy())
        swap_weights.append(w_swap.cpu().numpy())

    return (
        np.concatenate(all_targets, axis=0),
        np.concatenate(orig_probs, axis=0),
        np.concatenate(swap_probs, axis=0),
        np.concatenate(orig_weights, axis=0),
        np.concatenate(swap_weights, axis=0)
    )

def compute_metrics(y_true, y_prob, thresholds):
    thresholds = np.asarray(thresholds, dtype=np.float32)
    y_pred = (y_prob >= thresholds).astype(int)
    
    f1_list = []
    precision_list = []
    recall_list = []
    auroc_list = []
    
    for i in range(8):
        tp = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 1))
        fp = np.sum((y_true[:, i] == 0) & (y_pred[:, i] == 1))
        fn = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 0))
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        
        # AUROC (trapezoidal / rank-based sum)
        n_pos = np.sum(y_true[:, i] == 1)
        n_neg = np.sum(y_true[:, i] == 0)
        if n_pos > 0 and n_neg > 0:
            order = np.argsort(y_prob[:, i])
            rank = np.empty_like(order)
            rank[order] = np.arange(len(order))
            u_val = np.sum(rank[y_true[:, i] == 1]) - (n_pos * (n_pos - 1)) / 2.0
            auc = u_val / (n_pos * n_neg)
        else:
            auc = 0.5
        
        f1_list.append(float(f1))
        precision_list.append(float(prec))
        recall_list.append(float(rec))
        auroc_list.append(float(auc))
    
    macro_f1 = float(np.mean(f1_list))
    macro_auroc = float(np.mean(auroc_list))
    
    # Micro-F1
    total_tp = np.sum((y_true == 1) & (y_pred == 1))
    total_fp = np.sum((y_true == 0) & (y_pred == 1))
    total_fn = np.sum((y_true == 1) & (y_pred == 0))
    micro_f1 = float(2 * total_tp / (2 * total_tp + total_fp + total_fn)) if (2 * total_tp + total_fp + total_fn) > 0 else 0.0
    
    # Hamming Loss & Subset Accuracy
    hamming = float(np.mean(y_true != y_pred))
    subset_acc = float(np.mean(np.all(y_true == y_pred, axis=1)))
    
    return {
        "macro_f1": macro_f1,
        "macro_auroc": macro_auroc,
        "micro_f1": micro_f1,
        "hamming_loss": hamming,
        "subset_accuracy": subset_acc,
        "per_label_f1": f1_list,
        "per_label_precision": precision_list,
        "per_label_recall": recall_list,
        "per_label_auroc": auroc_list
    }

def optimize_thresholds_on_validation(y_val, p_val):
    best_thresh = []
    for i in range(8):
        best_f1 = -1.0
        best_t = 0.50
        for t in np.arange(0.01, 0.99, 0.01):
            pred = (p_val[:, i] >= t).astype(int)
            tp = np.sum((y_val[:, i] == 1) & (pred == 1))
            fp = np.sum((y_val[:, i] == 0) & (pred == 1))
            fn = np.sum((y_val[:, i] == 1) & (pred == 0))
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)
        best_thresh.append(best_t)
    return np.array(best_thresh, dtype=np.float32)


In [6]:
# 6. Verifikasi Validation Set & Penguncian Ambang Ensemble 3-Seed
val_probabilities = {}
LOCKED_THRESHOLDS = {}
y_val_gt = None

for seed in SEEDS:
    if seed in CHECKPOINTS:
        LOCKED_THRESHOLDS[seed] = RECORDED_VAL_THRESHOLDS[seed]

if valid_loader is not None and len(CHECKPOINTS) > 0:
    print("=" * 75)
    print("VERIFIKASI INFERENSI VALIDATION SET (525 PASIEN) & OPTIMASI ENSEMBLE")
    print("=" * 75)
    
    for seed in SEEDS:
        if seed not in CHECKPOINTS:
            continue
        ckpt_path = CHECKPOINTS[seed]
        model = LabelWiseRouterBilateralResNet50(num_labels=8).to(DEVICE)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
        model.load_state_dict(state_dict)
        
        y_v, p_v, _, _, _ = run_inference(model, valid_loader, DEVICE)
        y_val_gt = y_v
        val_probabilities[seed] = p_v
        
        res_v_05 = compute_metrics(y_v, p_v, np.full(8, 0.50))
        res_v_tuned = compute_metrics(y_v, p_v, LOCKED_THRESHOLDS[seed])
        print(f"Seed {seed} Validation -> Default 0.50: {res_v_05['macro_f1']:.4f} | Val-Locked: {res_v_tuned['macro_f1']:.4f}")
    
    # Optimasi ambang ensemble pada Validation Set
    available_seeds = list(val_probabilities.keys())
    p_val_ens = np.mean([val_probabilities[s] for s in available_seeds], axis=0)
    
    ens_thresh = optimize_thresholds_on_validation(y_val_gt, p_val_ens)
    LOCKED_THRESHOLDS["ensemble"] = ens_thresh
    
    res_ens_val_05 = compute_metrics(y_val_gt, p_val_ens, np.full(8, 0.50))
    res_ens_val_tuned = compute_metrics(y_val_gt, p_val_ens, ens_thresh)
    print(f"\n[VALIDATION RECORD] 3-Seed Ensemble:")
    print(f"  -> Val Macro-F1 (Default 0.50): {res_ens_val_05['macro_f1']:.4f}")
    print(f"  -> Val Macro-F1 (Val-Tuned)   : {res_ens_val_tuned['macro_f1']:.4f}")
    print(f"  -> Ambang Ensemble Terkunci   : {np.round(ens_thresh, 2).tolist()}")
else:
    print("Validation loader tidak aktif. Menggunakan ambang ensemble terstandarisasi.")
    LOCKED_THRESHOLDS["ensemble"] = np.array([0.05, 0.36, 0.80, 0.55, 0.50, 0.15, 0.60, 0.38], dtype=np.float32)

print("\nAmbang Keputusan Akhir Terkunci (Locked Thresholds for Test Set):")
for k, v in LOCKED_THRESHOLDS.items():
    print(f"  Model {str(k):<10}: {np.round(v, 2).tolist()}")


VERIFIKASI INFERENSI VALIDATION SET (525 PASIEN) & OPTIMASI ENSEMBLE
Seed 42 Validation -> Default 0.50: 0.6045 | Val-Locked: 0.5991
Seed 52 Validation -> Default 0.50: 0.6186 | Val-Locked: 0.6481
Seed 62 Validation -> Default 0.50: 0.6326 | Val-Locked: 0.6788

[VALIDATION RECORD] 3-Seed Ensemble:
  -> Val Macro-F1 (Default 0.50): 0.6443
  -> Val Macro-F1 (Val-Tuned)   : 0.7000
  -> Ambang Ensemble Terkunci   : [0.23000000417232513, 0.30000001192092896, 0.20999999344348907, 0.4099999964237213, 0.6899999976158142, 0.33000001311302185, 0.41999998688697815, 0.5099999904632568]

Ambang Keputusan Akhir Terkunci (Locked Thresholds for Test Set):
  Model 42        : [0.05999999865889549, 0.41999998688697815, 0.8399999737739563, 0.8500000238418579, 0.6800000071525574, 0.07000000029802322, 0.8700000047683716, 0.30000001192092896]
  Model 52        : [0.05000000074505806, 0.47999998927116394, 0.9200000166893005, 0.6600000262260437, 0.23000000417232513, 0.550000011920929, 0.5699999928474426, 0.37

In [7]:
# 7. Eksekusi Inferensi Test Set (525 Pasien Terkunci) - Satu Kali Eksekusi
print("=" * 75)
print("EKSEKUSI INFERENSI FINAL TEST SET (525 PASIEN) DENGAN AMBANG TERKUNCI")
print("=" * 75)

test_probabilities = {}
test_results_tuned = {}
test_results_05 = {}
test_swap_metrics = {}
y_test_gt = None

for seed in SEEDS:
    if seed not in CHECKPOINTS:
        print(f"Melewati Seed {seed} karena checkpoint tidak tersedia.")
        continue
        
    ckpt_path = CHECKPOINTS[seed]
    print(f"\n[Memproses Seed {seed}] Memuat checkpoint: {ckpt_path.name}")
    
    model = LabelWiseRouterBilateralResNet50(num_labels=8).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state_dict)
    
    y_true, p_orig, p_swap, w_orig, w_swap = run_inference(model, test_loader, DEVICE)
    y_test_gt = y_true
    test_probabilities[seed] = p_orig
    
    # Evaluasi Ambang Terkunci (Val-Locked Tuned)
    res_tuned = compute_metrics(y_true, p_orig, LOCKED_THRESHOLDS[seed])
    test_results_tuned[seed] = res_tuned
    
    # Evaluasi Ambang Default (0.50)
    res_05 = compute_metrics(y_true, p_orig, np.full(8, 0.50))
    test_results_05[seed] = res_05
    
    # Audit Pertukaran Mata di Test Set (Swap Invariance & Equivariance)
    delta_p = float(np.max(np.abs(p_orig - p_swap)))
    delta_w_left = np.abs(w_orig[:, 0, :] - w_swap[:, 1, :])
    delta_w_right = np.abs(w_orig[:, 1, :] - w_swap[:, 0, :])
    delta_w_bilat = np.abs(w_orig[:, 2, :] - w_swap[:, 2, :])
    delta_w = float(max(delta_w_left.max(), delta_w_right.max(), delta_w_bilat.max()))
    
    pred_orig = (p_orig >= LOCKED_THRESHOLDS[seed]).astype(int)
    pred_swap = (p_swap >= LOCKED_THRESHOLDS[seed]).astype(int)
    disagree = float(np.mean(pred_orig != pred_swap))
    
    test_swap_metrics[seed] = {
        "delta_p": delta_p,
        "delta_w": delta_w,
        "disagreement_rate": disagree
    }
    print(f"  -> Test Macro-F1 (Val-Locked Tuned) : {res_tuned['macro_f1']:.4f}")
    print(f"  -> Test Macro-F1 (Default 0.50)     : {res_05['macro_f1']:.4f}")
    print(f"  -> Test Macro-AUROC                 : {res_tuned['macro_auroc']:.4f}")
    print(f"  -> Test Micro-F1                    : {res_tuned['micro_f1']:.4f}")
    print(f"  -> Swap Delta p                     : {delta_p:.6f} (100% Invarian)")
    print(f"  -> Swap Delta w                     : {delta_w:.6f} (100% Ekuivarian)")

# Evaluasi 3-Seed Ensemble pada Test Set
available_seeds = list(test_probabilities.keys())
print(f"\n[Membuat Model Ensemble Test Set] Menggabungkan {len(available_seeds)} seed: {available_seeds}")

p_test_ensemble = np.mean([test_probabilities[s] for s in available_seeds], axis=0)
res_ensemble_tuned = compute_metrics(y_test_gt, p_test_ensemble, LOCKED_THRESHOLDS["ensemble"])
res_ensemble_05 = compute_metrics(y_test_gt, p_test_ensemble, np.full(8, 0.50))

print(f"  >>> 3-Seed Ensemble Test Macro-F1 (Val-Locked) : {res_ensemble_tuned['macro_f1']:.4f} <<<")
print(f"  >>> 3-Seed Ensemble Test Macro-F1 (Default 0.50): {res_ensemble_05['macro_f1']:.4f} <<<")
print(f"  >>> 3-Seed Ensemble Test Macro-AUROC           : {res_ensemble_tuned['macro_auroc']:.4f} <<<")
print(f"  >>> 3-Seed Ensemble Test Micro-F1              : {res_ensemble_tuned['micro_f1']:.4f} <<<")
print(f"  >>> 3-Seed Ensemble Test Hamming Loss          : {res_ensemble_tuned['hamming_loss']:.4f} <<<")
print(f"  >>> 3-Seed Ensemble Test Subset Accuracy       : {res_ensemble_tuned['subset_accuracy']:.4f} <<<")


EKSEKUSI INFERENSI FINAL TEST SET (525 PASIEN) DENGAN AMBANG TERKUNCI

[Memproses Seed 42] Memuat checkpoint: best_leber_a5_auxiliary_supervision_resnet50_bce_512_seed42.pt
  -> Test Macro-F1 (Val-Locked Tuned) : 0.5881
  -> Test Macro-F1 (Default 0.50)     : 0.6037
  -> Test Macro-AUROC                 : 0.8842
  -> Test Micro-F1                    : 0.6290
  -> Swap Delta p                     : 0.000000 (100% Invarian)
  -> Swap Delta w                     : 0.000000 (100% Ekuivarian)

[Memproses Seed 52] Memuat checkpoint: best_leber_a4_labelwise_router_resnet50_bce_512_seed52.pt
  -> Test Macro-F1 (Val-Locked Tuned) : 0.6460
  -> Test Macro-F1 (Default 0.50)     : 0.6274
  -> Test Macro-AUROC                 : 0.8946
  -> Test Micro-F1                    : 0.6656
  -> Swap Delta p                     : 0.000000 (100% Invarian)
  -> Swap Delta w                     : 0.000000 (100% Ekuivarian)

[Memproses Seed 62] Memuat checkpoint: best_leber_a4_labelwise_router_resnet50_bce_512_s

In [8]:
# 8. Tabel Rekapitulasi Komparasi Lengkap Test Set (Baseline A0 vs LEBER A4 vs Ensemble)

# Data historis Baseline A0 Test Set (ResNet-50 Bilateral BCE 512, N=525)
# Seed 42: F1=0.6122 | Seed 52: F1=0.6300 | Seed 62: F1=0.6096
BASELINE_A0_TEST = {
    "macro_f1_tuned": "0.6173 +/- 0.0091",
    "macro_f1_05": "0.5907 s.d. 0.6187",
    "macro_auroc": "0.8710 +/- 0.0057",
    "micro_f1": "0.6404 +/- 0.0039",
    "hamming_loss": "0.1159 +/- 0.0018",
    "subset_acc": "0.4184 +/- 0.0055",
    "swap": "Rentan (Delta p > 0)"
}

summary_rows = []
for s in available_seeds:
    t = test_results_tuned[s]
    d05 = test_results_05[s]
    summary_rows.append({
        "Model": f"LEBER A4 (Seed {s})",
        "Macro-F1 (Val-Locked)": f"{t['macro_f1']:.4f}",
        "Macro-F1 (0.50)": f"{d05['macro_f1']:.4f}",
        "Macro-AUROC": f"{t['macro_auroc']:.4f}",
        "Micro-F1": f"{t['micro_f1']:.4f}",
        "Hamming Loss": f"{t['hamming_loss']:.4f}",
        "Subset Acc": f"{t['subset_accuracy']:.4f}",
        "Swap (Delta p)": f"{test_swap_metrics[s]['delta_p']:.4f}"
    })

# Rata-rata 3 Seed LEBER A4
mean_f1_tuned = float(np.mean([test_results_tuned[s]['macro_f1'] for s in available_seeds]))
std_f1_tuned = float(np.std([test_results_tuned[s]['macro_f1'] for s in available_seeds]))
mean_f1_05 = float(np.mean([test_results_05[s]['macro_f1'] for s in available_seeds]))
std_f1_05 = float(np.std([test_results_05[s]['macro_f1'] for s in available_seeds]))
mean_auc = float(np.mean([test_results_tuned[s]['macro_auroc'] for s in available_seeds]))
std_auc = float(np.std([test_results_tuned[s]['macro_auroc'] for s in available_seeds]))
mean_micro = float(np.mean([test_results_tuned[s]['micro_f1'] for s in available_seeds]))
mean_hamming = float(np.mean([test_results_tuned[s]['hamming_loss'] for s in available_seeds]))
mean_subset = float(np.mean([test_results_tuned[s]['subset_accuracy'] for s in available_seeds]))

summary_rows.append({
    "Model": "LEBER A4 (Mean +/- Std)",
    "Macro-F1 (Val-Locked)": f"{mean_f1_tuned:.4f} +/- {std_f1_tuned:.4f}",
    "Macro-F1 (0.50)": f"{mean_f1_05:.4f} +/- {std_f1_05:.4f}",
    "Macro-AUROC": f"{mean_auc:.4f} +/- {std_auc:.4f}",
    "Micro-F1": f"{mean_micro:.4f}",
    "Hamming Loss": f"{mean_hamming:.4f}",
    "Subset Acc": f"{mean_subset:.4f}",
    "Swap (Delta p)": "0.0000"
})

# Ensemble 3-Seed LEBER A4
summary_rows.append({
    "Model": "LEBER A4 (3-Seed Ensemble)",
    "Macro-F1 (Val-Locked)": f"{res_ensemble_tuned['macro_f1']:.4f}",
    "Macro-F1 (0.50)": f"{res_ensemble_05['macro_f1']:.4f}",
    "Macro-AUROC": f"{res_ensemble_tuned['macro_auroc']:.4f}",
    "Micro-F1": f"{res_ensemble_tuned['micro_f1']:.4f}",
    "Hamming Loss": f"{res_ensemble_tuned['hamming_loss']:.4f}",
    "Subset Acc": f"{res_ensemble_tuned['subset_accuracy']:.4f}",
    "Swap (Delta p)": "0.0000"
})

# Baseline A0
summary_rows.append({
    "Model": "Baseline A0 (ResNet50 Bilateral)",
    "Macro-F1 (Val-Locked)": BASELINE_A0_TEST["macro_f1_tuned"],
    "Macro-F1 (0.50)": BASELINE_A0_TEST["macro_f1_05"],
    "Macro-AUROC": BASELINE_A0_TEST["macro_auroc"],
    "Micro-F1": BASELINE_A0_TEST["micro_f1"],
    "Hamming Loss": BASELINE_A0_TEST["hamming_loss"],
    "Subset Acc": BASELINE_A0_TEST["subset_acc"],
    "Swap (Delta p)": BASELINE_A0_TEST["swap"]
})

summary_df = pd.DataFrame(summary_rows)
print("=" * 85)
print("TABEL KOMPARASI PERFORMA FINAL PADA TEST SET (525 PASIEN TERKUNCI)")
print("=" * 85)
display(summary_df)

# Tabel Rincian F1 dan AUROC per Label
per_label_rows = []
for i, l in enumerate(LABELS):
    row = {
        "Label": l,
        "Penyakit": LABEL_NAMES[l],
        "Positif Test": int(np.sum(y_test_gt[:, i]))
    }
    for s in available_seeds:
        row[f"F1 S{s}"] = f"{test_results_tuned[s]['per_label_f1'][i]:.4f}"
    
    row["F1 Mean"] = f"{np.mean([test_results_tuned[s]['per_label_f1'][i] for s in available_seeds]):.4f}"
    row["F1 Ens (Val-Lock)"] = f"{res_ensemble_tuned['per_label_f1'][i]:.4f}"
    row["F1 Ens (0.50)"] = f"{res_ensemble_05['per_label_f1'][i]:.4f}"
    row["AUROC Ens"] = f"{res_ensemble_tuned['per_label_auroc'][i]:.4f}"
    per_label_rows.append(row)

per_label_df = pd.DataFrame(per_label_rows)
print("\n" + "=" * 85)
print("TABEL RINCIAN PERFORMA PER-LABEL PADA TEST SET")
print("=" * 85)
display(per_label_df)


TABEL KOMPARASI PERFORMA FINAL PADA TEST SET (525 PASIEN TERKUNCI)


,Model,Macro-F1 (Val-Locked),Macro-F1 (0.50),Macro-AUROC,Micro-F1,Hamming Loss,Subset Acc,Swap (Delta p)
0,LEBER A4 (Seed 42),0.5881,0.6037,0.8842,0.6290,0.1188,0.4343,0.0000
1,LEBER A4 (Seed 52),0.6460,0.6274,0.8946,0.6656,0.0990,0.5124,0.0000
2,LEBER A4 (Seed 62),0.6407,0.6243,0.9018,0.6793,0.0929,0.5390,0.0000
3,LEBER A4 (Mean +/- Std),0.6249 +/- 0.0261,0.6185 +/- 0.0105,0.8935 +/- 0.0072,0.6580,0.1036,0.4952,0.0000
4,LEBER A4 (3-Seed Ensemble),0.6626,0.6473,0.9178,0.6930,0.0907,0.5410,0.0000
5,Baseline A0 (ResNet50 Bilateral),0.6173 +/- 0.0091,0.5907 s.d. 0.6187,0.8710 +/- 0.0057,0.6404 +/- 0.0039,0.1159 +/- 0.0018,0.4184 +/- 0.0055,Rentan (Delta p > 0)



TABEL RINCIAN PERFORMA PER-LABEL PADA TEST SET


,Label,Penyakit,Positif Test,F1 S42,F1 S52,F1 S62,F1 Mean,F1 Ens (Val-Lock),F1 Ens (0.50),AUROC Ens
0,N,Normal,173,0.6841,0.6887,0.6792,0.6840,0.6891,0.5979,0.8631
1,D,Diabetes,169,0.7122,0.7255,0.7561,0.7313,0.7755,0.7443,0.8859
2,G,Glaucoma,32,0.3721,0.5000,0.5306,0.4676,0.5846,0.5385,0.9520
3,C,Cataract,32,0.7931,0.7869,0.7500,0.7767,0.7619,0.7869,0.9764
4,A,AMD,24,0.4865,0.6250,0.6667,0.5927,0.5556,0.6512,0.9637
5,H,Hypertension,15,0.2333,0.3750,0.2759,0.2947,0.4138,0.3478,0.8799
6,M,Myopia,26,0.8750,0.8750,0.8511,0.8670,0.8980,0.8980,0.9966
7,O,Others,147,0.5488,0.5917,0.6159,0.5855,0.6221,0.6139,0.8246


In [9]:
# 9. Penyimpanan Berkas Artefak Evaluasi Final Test Set
summary_df.to_csv(OUTPUT_DIR / "test_summary_metrics.csv", index=False)
per_label_df.to_csv(OUTPUT_DIR / "test_per_label_metrics.csv", index=False)

# Simpan probabilitas dan ground truth target
np.save(OUTPUT_DIR / "test_probabilities_ensemble.npy", p_test_ensemble)
np.save(OUTPUT_DIR / "test_targets.npy", y_test_gt)
for s in available_seeds:
    np.save(OUTPUT_DIR / f"test_probabilities_seed{s}.npy", test_probabilities[s])

with open(OUTPUT_DIR / "test_swap_metrics.json", "w") as f:
    json.dump(test_swap_metrics, f, indent=2)

with open(OUTPUT_DIR / "locked_thresholds_used.json", "w") as f:
    json.dump({
        str(k): [float(x) for x in v] for k, v in LOCKED_THRESHOLDS.items()
    }, f, indent=2)

with open(OUTPUT_DIR / "test_ensemble_metrics.json", "w") as f:
    json.dump({
        "ensemble_macro_f1_tuned": res_ensemble_tuned["macro_f1"],
        "ensemble_macro_f1_05": res_ensemble_05["macro_f1"],
        "ensemble_macro_auroc": res_ensemble_tuned["macro_auroc"],
        "ensemble_micro_f1": res_ensemble_tuned["micro_f1"],
        "ensemble_hamming_loss": res_ensemble_tuned["hamming_loss"],
        "ensemble_subset_accuracy": res_ensemble_tuned["subset_accuracy"],
        "per_label": {
            l: {
                "f1_tuned": float(res_ensemble_tuned["per_label_f1"][i]),
                "f1_05": float(res_ensemble_05["per_label_f1"][i]),
                "precision": float(res_ensemble_tuned["per_label_precision"][i]),
                "recall": float(res_ensemble_tuned["per_label_recall"][i]),
                "auroc": float(res_ensemble_tuned["per_label_auroc"][i])
            } for i, l in enumerate(LABELS)
        }
    }, f, indent=2)

print("\nSeluruh artefak evaluasi final Test Set berhasil disimpan di:")
print(OUTPUT_DIR)
print("- test_summary_metrics.csv")
print("- test_per_label_metrics.csv")
print("- test_ensemble_metrics.json")
print("- test_probabilities_ensemble.npy")
print("- test_swap_metrics.json")
print("- locked_thresholds_used.json")
print("\nEvaluasi Final Test Set Selesai 100%! Data siap digunakan untuk Bab 4 Naskah Skripsi.")



Seluruh artefak evaluasi final Test Set berhasil disimpan di:
/kaggle/working/leber_a4_final_test_evaluation
- test_summary_metrics.csv
- test_per_label_metrics.csv
- test_ensemble_metrics.json
- test_probabilities_ensemble.npy
- test_swap_metrics.json
- locked_thresholds_used.json

Evaluasi Final Test Set Selesai 100%! Data siap digunakan untuk Bab 4 Naskah Skripsi.
